In [ ]:
import numpy as np
import tensorflow as tf
import random
import datetime
import time
import json
import csv
from collections import deque
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import secrets

encryption_key = secrets.token_bytes(16)

class PPOAgent:
    def __init__(self, state_dim, action_dim, learning_rate=0.001, gamma=0.95, clip_ratio=0.2):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.clip_ratio = clip_ratio

        self.policy = self.build_policy_network()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(self.state_dim,)),
            tf.keras.layers.Dense(self.action_dim, activation='softmax')
        ])
        return model

    def get_action_prob(self, state):
        return self.policy.predict(np.array([state]))[0]

    def define_reward_function(self):


    def train(self, states, actions, advantages):
        actions = tf.keras.utils.to_categorical(actions, self.action_dim)
        advantages = np.array(advantages)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - self.clip_ratio, 1 + self.clip_ratio)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

In [ ]:
import random
import csv

def generate_random_binary_values(num_values):
    random_values = [random.randint(0, 1) for _ in range(num_values)]
    return random_values

# Generate 100 random binary values (0 or 1)
random_binary_values = generate_random_binary_values(100)

# Save to CSV
csv_file_path = 'random_binary_values.csv'
with open(csv_file_path, 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['Value'])
    csv_writer.writerows([[value] for value in random_binary_values])

print(f"Random binary values saved to {csv_file_path}")


Random binary values saved to random_binary_values.csv


In [ ]:
import random
import csv

def generate_random_numbers(num_numbers):
    random_numbers = [random.randint(0, 255) for _ in range(num_numbers)]
    return random_numbers

# Generate 100 random numbers between 10 and 100
random_numbers = generate_random_numbers(100)


# Save to CSV
csv_file_path = 'random_numbers.csv'
with open(csv_file_path, 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['Number'])
    csv_writer.writerows([[number] for number in random_numbers])

print(f"Random numbers saved to {csv_file_path}")



Random numbers saved to random_numbers.csv


In [ ]:
class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
            # # clip ratio hum alag se define kr sakte hai

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),  # Assuming 4 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]):
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained policy network
        state = np.array(list(input_data.values())).reshape(1, -1)
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())

        return 'ON' if action == 1 else 'OFF'

    def define_advantage(self, confidence: float) -> float:

        return confidence

# dummy dataset
ppo_agent = AIAgent(model_type='ReinforcementLearning')

# Train the PPO model
ppo_agent.train_rl(dummy_rl_dataset)

current_state = {'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}
action = ppo_agent.act(current_state)
print(f"Recommended action based on PPO: {action}")

In [ ]:
class Light:
    def __init__(self, RGB: bool, Ambient: bool, location: str, *args, **kwargs):
        """
        Initialize the Light object with specified parameters.

        Args:
            RGB (bool): Whether the light is an RGB light.
            Ambient (bool): Whether the light is an ambient light.
            location (str): Location of the light.
            *args, **kwargs: Additional arguments for specific light types.
        """

        self.RGB = RGB
        self.Ambient = Ambient
        self.location = location
        self.light = None  # Initialize light state as None (off)

    def control_normal_light(self, value: int):
        """
        Control the state of a normal light.

        Args:
            value (int): 0 for off, 1 for on.
        """

        if not self.RGB and not self.Ambient:
            if value in [0, 1]:
                self.light = value
                print(f"Normal light in {self.location} turned {'off' if not value else 'on'}")
            else:
                print(f"Invalid value for normal light control: {value}")
        else:
            print(f"This method is not applicable for {self.get_type()} light.")

    def control_ambient_light(self, brightness: int):
        """
        Control the state of an ambient light.

        Args:
            brightness (int): Brightness level (0-100).
        """

        if self.Ambient:
            if 0 <= brightness <= 100:
                self.light = brightness
                print(f"Ambient light in {self.location} set to brightness {brightness}")
            else:
                print(f"Invalid brightness value for ambient light: {brightness}")
        else:
            print(f"This method is not applicable for {self.get_type()} light.")

    def control_rgb_light(self, r: int, g: int, b: int):
        """
        Control the state of an RGB light.

        Args:
            r (int): Red value (0-255).
            g (int): Green value (0-255).
            b (int): Blue value (0-255).
        """

        if self.RGB:
            if 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
                self.light = {"r": r, "g": g, "b": b}
                print(f"RGB light in {self.location} set to color ({r}, {g}, {b})")
            else:
                print(f"Invalid color values for RGB light: (r: {r}, g: {g}, b: {b})")
        else:
            print(f"This method is not applicable for {self.get_type()} light.")

    def fetch_status(self) -> dict:
        """
        Fetch the current status of the light.

        Returns:
            dict: Dictionary containing light status details.
        """

        status = {"location": self.location}
        if self.RGB:
            status["type"] = "RGB"
            status["color"] = self.light
        elif self.Ambient:
            status["type"] = "Ambient"
            status["brightness"] = self.light
        else:
            status["type"] = "Normal"
            status["state"] = "On" if self.light else "Off"
        return status

    def get_type(self) -> str:
        """
        Get the type of the light.

        Returns:
            str: Light type (Normal, Ambient, or RGB).
        """

        if self.RGB:
            return "RGB"
        elif self.Ambient:
            return "Ambient"
        else:
            return "Normal"

# Example usage:
normal_light = Light(RGB=False, Ambient=False, location="Living room")
normal_light.control_normal_light(1)  # Turn on normal light

ambient_light = Light(RGB=False, Ambient=True, location="Bedroom")
ambient_light.control_ambient_light(50)  # Set ambient light brightness to 50

rgb_light = Light(RGB=True, Ambient=False, location="Kitchen")
rgb_light.control_rgb_light(255, 0, 0)  # Set RGB light to red



Normal light in Living room turned on
Ambient light in Bedroom set to brightness 50
RGB light in Kitchen set to color (255, 0, 0)


In [ ]:
import csv
from datetime import datetime, timedelta
from typing import List, Dict, Union, Any, Tuple
class Aggregator:
    def __init__(self, time, motion, TV_status, *args, **kwargs):
        self.time = time
        self.motion = motion
        self.TV_status = TV_status
        self.lights = []
        self.ai_agent = None
        # ... (other relevant parameters)

    def define_light_objects(self, lights_info: List[Dict[str, Union[bool, bool, str]]]):
        self.lights = [Light(**light_info) for light_info in lights_info]

    def define_ai_object(self, ai_params: dict) -> AIAgent:
        self.ai_agent = AIAgent(**ai_params)  # Assuming AIAgent class is defined elsewhere
        return self.ai_agent

    def invoke_light_control(self, light_obj: Light):
        # Implement logic to determine appropriate control function based on light type
        if light_obj.RGB:
            light_obj.control_rgb_light(*light_obj.light)  # Assuming light state is a tuple for RGB
        elif light_obj.Ambient:
            light_obj.control_ambient_light(light_obj.light)
        else:
            light_obj.control_normal_light(light_obj.light)

    def save_ai_output_to_csv(self, ai_output: str):
        today = datetime.now().date()
        filename = f"ai_output_{today}.csv"
        try:
            with open(filename, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow([self.time, ai_output])
        except FileNotFoundError:
            with open(filename, 'w', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(["Time", "AI Output"])
                writer.writerow([self.time, ai_output])

        # Delete files older than 15 days
        cutoff_date = today - timedelta(days=15)
        for file in os.listdir():
            if file.startswith("ai_output_") and datetime.strptime(file[11:21], "%Y-%m-%d").date() < cutoff_date:
                os.remove(file)

    def compute_difference(self, previous_state: dict, current_state: dict) -> bool:
        # Implement logic to compare states and determine manual override
        # ...
        return True  # Replace with actual logic

    def compute_future_difference(self, current_state: dict, next_predicted_state: dict) -> bool:
        # Implement logic to compare current and predicted states and decide on signal
        # ...
        return True  # Replace with actual logic

    def send_signal(self):
        # Implement signal sending functionality
        pass


In [ ]:
import tensorflow as tf  # Assuming TensorFlow for deep learning models

class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type
        if model_type == "deep_learning":
            self.model = None  # Initialize model later when training

    def train(self, dataset: List[Dict[str, Any]]):
      data = pd.read_csv("normal_light.csv")

      # Preprocess the data
      data["time"] = pd.to_datetime(data["time"]).dt.hour  # Extract hour from time
      data["location"] = data["location"].astype("category").cat.codes  # Encode categorical location
      data["light_type"] = data["light_type"].astype("category").cat.codes  # Encode categorical light type

      # Separate features (X) and target variables (y)
      X = data.drop(["light_type", "pred_on"], axis=1)
      y = data["pred_on"]

      # Split data into training and testing sets
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

      # Create the neural network model
      model = Sequential()
      model.add(InputLayer(input_shape=(X_train.shape[1],)))
      model.add(Dense(64, activation="relu"))  # Hidden layer with 64 neurons
      model.add(Dense(1, activation="sigmoid"))  # Output layer with sigmoid activation

      # Compile the model
      model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

      # Train the model
      model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))

    def predict(self, input_data: Any) -> str:
        if self.model_type == "deep_learning":
            if self.model is None:
                raise ValueError("Model not trained yet.")
            prediction = self.model.predict(input_data)  # Get model prediction
            # Process prediction output to generate state prediction
            return "Predicted state"  # Replace with actual state prediction
        else:
            # Implement prediction logic for non-deep learning models
            return "Predicted state"  # Replace with actual logic


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, InputLayer

# Load the CSV data
data = pd.read_csv("normal_light.csv")

# Preprocess the data
data["time"] = pd.to_datetime(data["time"]).dt.hour  # Extract hour from time
data["location"] = data["location"].astype("category").cat.codes  # Encode categorical location
data["light_type"] = data["light_type"].astype("category").cat.codes  # Encode categorical light type

# Separate features (X) and target variables (y)
X = data.drop(["light_type", "pred_on"], axis=1)
y = data["pred_on"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create the neural network model
model = Sequential()
model.add(InputLayer(input_shape=(X_train.shape[1],)))
model.add(Dense(64, activation="relu"))  # Hidden layer with 64 neurons
model.add(Dense(1, activation="sigmoid"))  # Output layer with sigmoid activation

# Compile the model
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

# Train the model
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))

# Evaluate the model's performance
loss, accuracy = model.evaluate(X_test, y_test)
print("Test loss:", loss)
print("Test accuracy:", accuracy)

# # Use the model to make predictions on new data
# new_data = pd.DataFrame({
#     "time": 20,
#     "location": "kitchen",  # Assuming numerical encoding for locations
#     # "light_type": [1],  # Assuming numerical encoding for light types
#     "prev_on": 1,
#     # "previously_intensity": [50],
#     "motion": 1,
#     "tv_on": 0
# })
# predictions = model.predict(new_data)
# print("Predictions:", predictions)
